# Seasonal variation of the CLCT cloud/no-cloud threshold

Operational **KENDA-CH1 deterministic first guess** against the **NWCSAF cloud mask**, one full year,
at the four synoptic hours. For every timestep the threshold `tau` that maximises the Peirce skill
score (POD - POFD) is fitted, and the question is simply: **how much does `tau` move?**

Anything binarising a continuous CLCT field has to pick a threshold. If `tau` turns out to be stable,
a fixed value is defensible. If it swings with season or time of day, a fixed value silently injects a
seasonal or diurnal error into whatever it feeds.

**One thing has to be read carefully.** NWCSAF applies a *different sequence of spectral tests* by day,
at twilight and at night (CMA product description: day = solar elevation > 10 deg, twilight -3..10,
night < -3). The detection sensitivity therefore changes with illumination, and so does `tau`. Over a
year:

| | 00 UTC | 06 UTC | 12 UTC | 18 UTC |
|---|---|---|---|---|
| regime | night all year | night -> twilight -> **day** -> twilight -> night | day all year | night -> twilight -> **day** -> twilight -> night |

So **00 and 12 UTC are directly interpretable** -- they stay in one regime, and any seasonal movement
in `tau` is real. **06 and 18 UTC sweep through all three regimes**, which on its own manufactures a
smooth seasonal arc in `tau`. Both are computed; every point is coloured by the regime the satellite
actually used, so the two readings can be told apart.

Upstream processing is unchanged from `CLCT_threshold_corrected.ipynb` and `docs/conventions.md`:
conservative regridding, satellite slot `T-15min`, parallax correction, snow/ice screening.

In [ ]:
import os
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import eccodes
import matplotlib.pyplot as plt

# =============================================================================
# Config
# =============================================================================
NWCSAF_DIR = "/scratch/mch/jdelbeke/nwcsaf_year"
KENDA_DIR_T = "/store_new/mch/msopr/osm/KENDA-CH1/FG{yy}/det"
CACHE_DIR = "/scratch/mch/jdelbeke"
WEIGHTS_CACHE = f"{CACHE_DIR}/regrid_weights_icon_nwcsaf.npz"   # shared, unchanged
FIELD_CACHE = f"{CACHE_DIR}/tau_seasonal_fields.npz"            # the expensive pass
FIGDIR = Path("./figures"); FIGDIR.mkdir(parents=True, exist_ok=True)

# One year ending at the newest timestep with BOTH datasets present. The satellite
# archive is current to within a few hours; KENDA FG26 ended at 2026-09-18 12 UTC,
# whose T-15 slot did not exist yet, so the last complete day is 2026-09-17.
# The requested window is one year back from the newest timestep present in BOTH
# datasets. KENDA FG26 ends 2026-09-18 12 UTC, whose T-15 slot does not exist yet,
# so the last complete day is 2026-09-17.
#
# Extended back to a continuous ~23-month record. Two reasons it is not just the
# one requested year: the archive is only 52-61% complete for Nov 2025 - Jan 2026
# (a 30-day outage from 17 Nov, another 15 days from 19 Jan), which is precisely
# where tau turns out to move most; and the 2024-25 winter is 100% present, so it
# gives an independent check on whether the seasonal cycle repeats. It does.
WINDOWS = [(datetime(2024, 11, 1, 0), datetime(2026, 9, 17, 18), "Nov 2024 - Sep 2026")]
SYNOPTIC_HOURS = [0, 6, 12, 18]

TIMES, WINDOW_OF = [], {}
for _a, _b, _tag in WINDOWS:
    _d = _a
    while _d <= _b:
        if _d.hour in SYNOPTIC_HOURS:
            TIMES.append(_d); WINDOW_OF[_d] = _tag
        _d += timedelta(hours=1)
TIMES.sort()

NWCSAF_SLOT_LAG = timedelta(minutes=15)   # verified; docs/conventions.md
MIN_COVERAGE = 0.999
PARALLAX = True
CT_SNOW_ICE = (3, 4)          # Snow_over_land, Sea_ice -- the only screen
SAT_LON = 0.0
N_WORKERS = 8                 # the pass reads ~4.9 TB off /store; be a good citizen

# threshold grid: lower-inclusive, so "CLCT >= TAUS[k]" is exactly "bin >= k"
BIN_PER_PCT = 2
NBIN = 100 * BIN_PER_PCT + 1
TAUS = np.arange(NBIN) / BIN_PER_PCT
PLATEAU_TOL = 0.01            # tau values within this of max J count as "as good"

N_VERIFY = 20                 # files re-read via earthkit to prove the fast path agrees

def nwcsaf_file(t):
    return f"{NWCSAF_DIR}/MSG_nwcsaf_cosmo1eqc3km_{t:%Y%m%d%H%M}.nc"

def kenda_file(t):
    """Year-aware: the window spans FG25 and FG26."""
    return f"{KENDA_DIR_T.format(yy=t.strftime('%y'))}/lff{t:%Y%m%d%H}"

for _a, _b, _tag in WINDOWS:
    _n = sum(1 for t in TIMES if _a <= t <= _b)
    print(f"{_tag:>17s}: {_n:5d} timesteps, {_a:%Y-%m-%d} .. {_b:%Y-%m-%d}")
print(f"{'total':>17s}: {len(TIMES):5d} timesteps at {SYNOPTIC_HOURS} UTC")

## 1. Reading, regridding and correcting

Same operations as the experiment notebooks. The only change is how the GRIB is read: see the note on the fast path.

In [ ]:
# --- the fast GRIB read ------------------------------------------------------
# CLCT is ONE message out of ~1418, and it sits ~97.5% of the way into a 3.3 GB
# file, so there is no avoiding the scan -- but there is a lot to avoid in how
# the scan is done. `earthkit ... .to_xarray()` costs 12.2 s per file because it
# builds full metadata for every message; reading the values straight out of
# eccodes and stopping at the first CLCT costs 2.2 s. Verified bitwise identical
# (raw values AND downstream tau*/J to 5 decimals) on files from both FG25 and
# FG26, winter and summer; N_VERIFY files are re-checked during the real pass.
#
# Searching by shortName is deliberate and must stay: the message INDEX is not
# stable (1381 of 1419 in January, 1375 of 1417 in July), so a hardcoded offset
# would be a silent, seasonally varying corruption.
N_ICON_CELLS = 1147980

def read_clct(path):
    found = None
    with open(path, "rb") as f:
        while True:
            gid = eccodes.codes_grib_new_from_file(f)
            if gid is None:
                break
            if eccodes.codes_get(gid, "shortName") == "CLCT":
                if found is not None:
                    eccodes.codes_release(gid)
                    raise RuntimeError(f"more than one CLCT message in {path}")
                eccodes.codes_set(gid, "missingValue", np.nan)
                found = eccodes.codes_get_values(gid).astype("float64")
            eccodes.codes_release(gid)
    if found is None:
        raise RuntimeError(f"no CLCT message in {path}")
    if found.size != N_ICON_CELLS:
        raise RuntimeError(f"{path}: {found.size} values, expected {N_ICON_CELLS}")
    return found


def read_clct_earthkit(path):
    """The original path, kept only to verify the fast one on a sample."""
    import earthkit.data as ekd
    fl = ekd.from_source("file", path).to_fieldlist()
    xa = fl.sel({"parameter.variable": "CLCT"})[0].to_xarray()
    nm = "CLCT" if "CLCT" in xa else list(xa.data_vars)[0]
    return np.asarray(xa[nm].values, dtype="float64").ravel()


# --- regridding (weights shared with the other notebooks, never rebuilt here) --
_z = np.load(WEIGHTS_CACHE)
SUB, TI, CI, W, COVERAGE = _z["sub"], _z["ti"], _z["ci"], _z["w"], _z["coverage"]
SHAPE = COVERAGE.shape
IN_DOMAIN = COVERAGE >= MIN_COVERAGE

def remap(values):
    v = np.asarray(values, dtype="float64").ravel()[SUB]
    ok = np.isfinite(v[TI])
    n = SHAPE[0] * SHAPE[1]
    num = np.bincount(CI[ok], weights=W[ok] * v[TI][ok], minlength=n)
    den = np.bincount(CI[ok], weights=W[ok], minlength=n)
    out = np.full(n, np.nan)
    good = (COVERAGE.ravel() >= MIN_COVERAGE) & (den > 0)
    out[good] = num[good] / den[good]
    return out.reshape(SHAPE)

print(f"target grid {SHAPE}, {int(IN_DOMAIN.sum())} cells inside the ICON domain")

In [ ]:
# --- grid geometry, for parallax --------------------------------------------
with xr.open_dataset(nwcsaf_file(TIMES[0] - NWCSAF_SLOT_LAG)) as _ds:
    _lo = np.asarray(_ds["longitude"].values); _la = np.asarray(_ds["latitude"].values)
LON = _lo[0, :] if _lo.ndim == 2 else _lo
LAT = _la[:, 0] if _la.ndim == 2 else _la
DLON = float(np.median(np.diff(LON))); DLAT = float(np.median(np.diff(LAT)))
LON2, LAT2 = np.meshgrid(LON, LAT)

R_E, R_S = 6371.0, 42164.0
_la_r, _dlo_r = np.deg2rad(LAT2), np.deg2rad(LON2 - SAT_LON)
PSI = np.arccos(np.cos(_la_r) * np.cos(_dlo_r))
VZA = np.arctan2(R_S * np.sin(PSI), R_S * np.cos(PSI) - R_E)
BRG = np.arctan2(np.sin(-_dlo_r), -np.sin(_la_r) * np.cos(-_dlo_r))
DLAT_PER_KM = np.tan(VZA) * np.cos(BRG) / 111.32
DLON_PER_KM = np.tan(VZA) * np.sin(BRG) / (111.32 * np.cos(_la_r))

# model orography on the target grid, for the parallax of clear pixels
_hs = None
with open(kenda_file(TIMES[0]), "rb") as _f:
    while True:
        _g = eccodes.codes_grib_new_from_file(_f)
        if _g is None: break
        if eccodes.codes_get(_g, "shortName") == "HSURF":
            _hs = eccodes.codes_get_values(_g).astype("float64")
        eccodes.codes_release(_g)
        if _hs is not None: break
if _hs is None:
    raise RuntimeError("HSURF not found in the KENDA lff file")
HSURF = remap(_hs)
HSURF_KM = np.where(np.isfinite(HSURF), HSURF, 0.0) / 1000.0
print(f"viewing zenith at domain centre "
      f"{np.rad2deg(VZA[SHAPE[0]//2, SHAPE[1]//2]):.1f} deg; HSURF "
      f"{np.nanmin(HSURF):.0f}-{np.nanmax(HSURF):.0f} m")

In [ ]:
def decode_flags(da):
    a = np.asarray(da.values)
    ai = np.where(np.isfinite(a), a, 0).astype("int64")
    mn = da.attrs["flag_meanings"].split()
    vv = np.asarray(da.attrs["flag_values"]).astype("int64")
    mk = np.asarray(da.attrs.get("flag_mask", da.attrs.get("flag_masks", vv))).astype("int64")
    out, seen = {}, {}
    for m, v, k in zip(mn, vv, mk):
        seen[m] = seen.get(m, 0) + 1
        out[m if seen[m] == 1 else f"{m}__{seen[m]}"] = (ai & int(k)) == int(v)
    return out


def parallax_correct(cma, alti, no_method):
    """Displace the mask onto true ground positions. A cell that only an
    uncorrectable cloudy pixel could occupy is unobserved (NaN), not clear."""
    ny, nx = cma.shape
    cloudy, clear = cma == 1, cma == 0
    h_km = alti / 1000.0
    ok_h = np.isfinite(h_km) & ~no_method
    movable, lost = cloudy & ok_h, cloudy & ~ok_h

    def scatter(mask, z_km):
        i0, j0 = np.nonzero(mask)
        lat_t = LAT2[i0, j0] + DLAT_PER_KM[i0, j0] * z_km[i0, j0]
        lon_t = LON2[i0, j0] + DLON_PER_KM[i0, j0] * z_km[i0, j0]
        i1 = np.rint((lat_t - LAT[0]) / DLAT).astype(int)
        j1 = np.rint((lon_t - LON[0]) / DLON).astype(int)
        keep = (i1 >= 0) & (i1 < ny) & (j1 >= 0) & (j1 < nx)
        cnt = np.zeros((ny, nx), dtype=np.int32)
        np.add.at(cnt, (i1[keep], j1[keep]), 1)
        return cnt

    n_cloud = scatter(movable, np.where(ok_h, h_km, 0.0))
    n_clear = scatter(clear, HSURF_KM)
    n_lost = scatter(lost, HSURF_KM)
    out = np.full((ny, nx), np.nan)
    out[n_clear > 0] = 0.0
    out[n_cloud > 0] = 1.0                      # cloud wins a collision
    out[(n_lost > 0) & (n_cloud == 0)] = np.nan
    return out


def read_obs(slot):
    """Parallax-corrected, snow/ice-screened mask, plus regime and diagnostics."""
    with xr.open_dataset(nwcsaf_file(slot)) as ds:
        cma = np.asarray(ds["cma"].squeeze().values, dtype="float64")
        cma[cma == 255] = np.nan
        ct = np.asarray(ds["ct"].squeeze().values, dtype="float64")
        alti = np.asarray(ds["ctth_alti"].squeeze().values, dtype="float64")
        lo, hi = ds["ctth_alti"].attrs.get("valid_range", (-2000.0, 25000.0))
        alti[(alti < lo) | (alti > hi)] = np.nan
        meth = decode_flags(ds["ctth_method"])
        cond = decode_flags(ds["ctth_conditions"])
    no_method = meth.get("No_reliable_method", np.zeros(SHAPE, bool))
    obs = parallax_correct(cma, alti, no_method) if PARALLAX else cma
    snow = np.isin(ct, CT_SNOW_ICE)
    obs = np.where(snow, np.nan, obs)
    regime = max(("day", "twilight", "night"),
                 key=lambda r: int((cond.get(r, np.zeros(SHAPE, bool)) & IN_DOMAIN).sum()))
    return obs, regime, float((snow & IN_DOMAIN).sum() / IN_DOMAIN.sum())

## 2. The expensive pass, done once

Reads ~4.9 TB off `/store` to pull one message per file, regrids, corrects, and caches the result at ~1.2 GB. Every analysis after this is instant.

In [ ]:
import sys, traceback
from concurrent.futures import ProcessPoolExecutor, as_completed

SIG = "|".join([str(WINDOWS), str(SYNOPTIC_HOURS), str(MIN_COVERAGE),
                str(PARALLAX), str(NWCSAF_SLOT_LAG), str(CT_SNOW_ICE)])

def build_one(t):
    """CLCT regridded onto the satellite grid, plus the corrected mask."""
    slot = t - NWCSAF_SLOT_LAG
    if not os.path.exists(nwcsaf_file(slot)):
        raise FileNotFoundError(f"no satellite slot {slot:%Y-%m-%d %H:%M}")
    clct = remap(read_clct(kenda_file(t)))
    obs, regime, snow_frac = read_obs(slot)
    return (t, clct.astype("float32"),
            np.where(np.isfinite(obs), obs, -1).astype("int8"), regime, snow_frac)

def _job(t):
    try:
        return build_one(t), None
    except Exception as e:
        return None, f"{t:%Y-%m-%d %H}: {type(e).__name__}: {e}"

_cached = None
if os.path.exists(FIELD_CACHE):
    _z2 = np.load(FIELD_CACHE, allow_pickle=True)
    if str(_z2["signature"]) == SIG:
        _cached = _z2
        print(f"field cache hit: {len(_z2['times'])} timesteps from {FIELD_CACHE}")
    else:
        print("field cache was built with different settings -- rebuilding")

if _cached is not None:
    TIMES_used = [datetime.fromisoformat(str(x)) for x in _cached["times"]]
    CLCT_ALL, OBS_ALL = _cached["clct"], _cached["obs"]
    REGIME = list(_cached["regime"]); SNOW = list(_cached["snow"])
    failed = []
else:
    import time as _time
    _t0 = _time.time()
    results, failed = [], []
    print(f"reading {len(TIMES)} timesteps on {N_WORKERS} workers "
          f"(~{len(TIMES) * 3.3 / 1000:.1f} TB off /store)", flush=True)
    with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_job, t) for t in TIMES]
        for done, fu in enumerate(as_completed(futs), 1):
            res, err = fu.result()
            if err:
                failed.append(err); print(f"  SKIP {err}", flush=True)
            else:
                results.append(res)
            if done % 100 == 0:
                el = _time.time() - _t0
                print(f"  {done}/{len(TIMES)}  {el/60:.1f} min elapsed, "
                      f"~{el/done*(len(TIMES)-done)/60:.1f} min left", flush=True)
    results.sort(key=lambda r: r[0])
    TIMES_used = [r[0] for r in results]
    CLCT_ALL = np.stack([r[1] for r in results])
    OBS_ALL = np.stack([r[2] for r in results])
    REGIME = [r[3] for r in results]; SNOW = [r[4] for r in results]
    np.savez(FIELD_CACHE, times=np.array([x.isoformat() for x in TIMES_used]),
             clct=CLCT_ALL, obs=OBS_ALL, regime=np.array(REGIME),
             snow=np.array(SNOW), signature=np.array(SIG))
    print(f"\n{len(TIMES_used)}/{len(TIMES)} timesteps in "
          f"{(_time.time()-_t0)/60:.1f} min; cache -> {FIELD_CACHE} "
          f"({os.path.getsize(FIELD_CACHE)/1e9:.2f} GB)")

print(f"{len(TIMES_used)} timesteps usable, {len(failed)} missing")

In [ ]:
# --- prove the fast GRIB read on a random sample of the ACTUAL run -----------
# Spot-checking 5 files before the run showed the two paths agree; this checks a
# sample of the files that were really used, so the claim covers this dataset
# rather than a handful chosen by hand.
if _cached is not None:
    print('field cache was reused -- the read path did not run, nothing to verify')
_rng = np.random.default_rng(20260918)
_pick = ([] if _cached is not None else
         _rng.choice(len(TIMES_used), size=min(N_VERIFY, len(TIMES_used)),
                     replace=False))
_bad = []
for _i in _pick:
    _t = TIMES_used[_i]
    _fast = read_clct(kenda_file(_t))
    _slow = read_clct_earthkit(kenda_file(_t))
    if not np.array_equal(_fast, _slow, equal_nan=True):
        _bad.append(_t)
if len(_pick):
    print(f"fast vs earthkit read, {len(_pick)} random files of the run: "
          f"{'ALL BITWISE IDENTICAL' if not _bad else f'MISMATCH at {_bad}'}")
if _bad:
    raise RuntimeError("the fast GRIB path does not reproduce earthkit -- stop here")

## 3. Fit `tau` per timestep

Exact histogram binning, so `CLCT >= TAUS[k]` is identically `bin >= k`. The plateau is every `tau` within `PLATEAU_TOL` of the best J -- it is the honest width of the answer, and at this sample size it is wide.

In [ ]:
def fit_tau(clct, obs_i8, mask=None):
    """tau maximising J = POD - POFD, plus the plateau of near-equal values."""
    base = IN_DOMAIN if mask is None else (IN_DOMAIN & mask)
    valid = base & (obs_i8 >= 0) & np.isfinite(clct)
    if valid.sum() < 1000:
        return dict(tau=np.nan, J=np.nan, pod=np.nan, pofd=np.nan,
                    lo=np.nan, hi=np.nan, obs_frac=np.nan, n=int(valid.sum()))
    b = np.clip(np.floor(np.where(np.isfinite(clct), clct, 0.0) * BIN_PER_PCT),
                0, NBIN - 1).astype(int)[valid]
    o = obs_i8[valid] == 1
    hc = np.bincount(b[o], minlength=NBIN).astype(np.int64)
    hk = np.bincount(b[~o], minlength=NBIN).astype(np.int64)
    pod = np.cumsum(hc[::-1])[::-1] / max(hc.sum(), 1)
    pofd = np.cumsum(hk[::-1])[::-1] / max(hk.sum(), 1)
    J = pod - pofd
    k = int(np.argmax(J))
    plat = TAUS[J >= J[k] - PLATEAU_TOL]
    return dict(tau=float(TAUS[k]), J=float(J[k]), pod=float(pod[k]),
                pofd=float(pofd[k]), lo=float(plat.min()), hi=float(plat.max()),
                obs_frac=float(o.mean()), n=int(valid.sum()))


rows = []
for i, t in enumerate(TIMES_used):
    clct, obs_i8 = CLCT_ALL[i].astype("float64"), OBS_ALL[i]
    r = fit_tau(clct, obs_i8)
    rows.append(dict(time=t, hour=t.hour, regime=REGIME[i],
                     window=WINDOW_OF.get(t, '?'),
                     tau_star=r["tau"], J=r["J"], POD=r["pod"], POFD=r["pofd"],
                     plateau_lo=r["lo"], plateau_hi=r["hi"],
                     obs_cloud_frac=r["obs_frac"], n_valid=r["n"],
                     model_mean_clct=float(np.nanmean(np.where(IN_DOMAIN, clct, np.nan))),
                     frac_snow_screened=SNOW[i],
                     frac_unobserved=float(1 - (obs_i8 >= 0)[IN_DOMAIN].mean())))
TAU = pd.DataFrame(rows).set_index("time").sort_index()
TAU.to_csv(FIGDIR / "clct_tau_seasonal.csv")
print(f"wrote {FIGDIR / 'clct_tau_seasonal.csv'}  ({len(TAU)} rows)")

print(f"\n{'hour':>5s} {'regime(s)':>22s} {'n':>5s} {'tau median':>11s} {'IQR':>13s} "
      f"{'range':>13s} {'plateau':>9s} {'J':>6s}")
for h in SYNOPTIC_HOURS:
    s = TAU[TAU.hour == h]
    rg = ", ".join(f"{k} {v}" for k, v in s.regime.value_counts().items())
    q1, q3 = s.tau_star.quantile([0.25, 0.75])
    print(f"{h:4d}Z {rg:>22s} {len(s):5d} {s.tau_star.median():10.1f}% "
          f"{q1:5.1f}-{q3:5.1f}% {s.tau_star.min():5.1f}-{s.tau_star.max():5.1f}% "
          f"{(s.plateau_hi - s.plateau_lo).median():8.1f}% {s.J.median():6.3f}")

print(f"\n{'regime':>9s} {'n':>5s} {'tau median':>11s} {'IQR':>13s} {'range':>13s}")
for rg in ("day", "twilight", "night"):
    s = TAU[TAU.regime == rg]
    if not len(s): continue
    q1, q3 = s.tau_star.quantile([0.25, 0.75])
    print(f"{rg:>9s} {len(s):5d} {s.tau_star.median():10.1f}% {q1:5.1f}-{q3:5.1f}% "
          f"{s.tau_star.min():5.1f}-{s.tau_star.max():5.1f}%")

## 4. The figures

In [ ]:
import matplotlib.dates as mdates
REGIME_COLOUR = {"day": "#E8A33D", "twilight": "#8C6BB1", "night": "#2C5985"}
HOUR_LABEL = {0: "00 UTC", 6: "06 UTC", 12: "12 UTC", 18: "18 UTC"}

fig, axes = plt.subplots(len(SYNOPTIC_HOURS), 1, figsize=(15, 12), sharex=True, sharey=True)
for ax, h in zip(axes, SYNOPTIC_HOURS):
    s = TAU[TAU.hour == h].sort_index()
    # Plot each window on its own. fill_between and plot both interpolate across
    # missing time, so drawing the whole series at once painted a plateau band
    # and a median line straight across the 7-month gap between the winter
    # backfill and the main year -- an eye-catching feature over a period with
    # no data at all.
    seen = set()
    for _a, _b, _tag in WINDOWS:
        w = s[(s.index >= _a) & (s.index <= _b)]
        if not len(w):
            continue
        # Reindex onto every day the window should contain, so the archive's own
        # outages (17 Nov - 16 Dec 2025, 19 Jan - 2 Feb 2026, and singles) become
        # NaN and break the band. Without this, fill_between spans them and paints
        # a plateau across days that were never observed.
        w = w.reindex(pd.date_range(w.index.min(), w.index.max(), freq="D"))
        ax.fill_between(w.index, w.plateau_lo, w.plateau_hi, color="#bbbbbb",
                        alpha=0.45, lw=0, zorder=1,
                        label=(f"plateau (J within {PLATEAU_TOL})"
                               if "plat" not in seen else None))
        seen.add("plat")
        for rg in ("day", "twilight", "night"):
            m = w.regime == rg
            if m.any():
                ax.plot(w.index[m], w.tau_star[m], "o", ms=3.2,
                        color=REGIME_COLOUR[rg], lw=0, zorder=3,
                        label=(rg if rg not in seen else None))
                seen.add(rg)
        mm = w.tau_star.resample("MS").median()
        ax.plot(mm.index + pd.Timedelta(days=15), mm.values, "-", color="k",
                lw=2.0, zorder=4,
                label=("monthly median" if "mm" not in seen else None))
        seen.add("mm")
    n_reg = s.regime.nunique()
    ax.set_title(f"{HOUR_LABEL[h]}   median {s.tau_star.median():.0f}%   "
                 f"range {s.tau_star.min():.0f}-{s.tau_star.max():.0f}%   "
                 + ("single regime -- seasonal movement is real"
                    if n_reg == 1 else
                    f"{n_reg} regimes -- part of the seasonal arc is the satellite"),
                 fontsize=10, loc="left")
    ax.set_ylabel("tau* [%]")
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 100)
axes[0].legend(fontsize=8, ncol=5, loc="lower left")
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
axes[-1].set_xlabel("date")
fig.suptitle("Optimal CLCT threshold (max Peirce skill score), KENDA-CH1 det FG vs NWCSAF, "
             f"{TAU.index[0]:%d %b %Y} - {TAU.index[-1]:%d %b %Y}", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(FIGDIR / "clct_tau_seasonal.png", dpi=120)
plt.show()
print(f"wrote {FIGDIR / 'clct_tau_seasonal.png'}")

In [ ]:
# --- diagnostics: is a tau excursion meteorology or sample composition? ------
fig, axes = plt.subplots(4, 1, figsize=(15, 11), sharex=True)
for h in SYNOPTIC_HOURS:
    s = TAU[TAU.hour == h].sort_index()
    axes[0].plot(s.index, s.tau_star, lw=0.8, alpha=0.85, label=HOUR_LABEL[h])
    axes[1].plot(s.index, s.obs_cloud_frac, lw=0.8, alpha=0.85)
    axes[2].plot(s.index, 100 * s.frac_snow_screened, lw=0.8, alpha=0.85)
    axes[3].plot(s.index, 100 * s.frac_unobserved, lw=0.8, alpha=0.85)
for ax, lab in zip(axes, ["tau* [%]", "observed cloud fraction",
                          "snow/ice screened [%]", "unobserved [%]"]):
    ax.set_ylabel(lab); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8, ncol=4, loc="lower left")
axes[0].set_title("tau* against the things that could move it for non-meteorological reasons",
                  fontsize=11, loc="left")
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
fig.tight_layout()
fig.savefig(FIGDIR / "clct_tau_seasonal_diagnostics.png", dpi=120)
plt.show()

# --- distributions: the direct answer to "how much does tau vary" -----------
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
data = [TAU[TAU.hour == h].tau_star.dropna() for h in SYNOPTIC_HOURS]
bp = axes[0].boxplot(data, tick_labels=[HOUR_LABEL[h] for h in SYNOPTIC_HOURS],
                     showfliers=False, patch_artist=True)
for p in bp["boxes"]: p.set_facecolor("#cfd8e3")
axes[0].set_ylabel("tau* [%]"); axes[0].grid(alpha=0.3, axis="y")
axes[0].set_title("by synoptic hour", fontsize=10, loc="left")

months = sorted(TAU.index.to_period("M").unique())
md_ = [TAU[TAU.index.to_period("M") == m].tau_star.dropna() for m in months]
bp2 = axes[1].boxplot(md_, tick_labels=[str(m)[-2:] for m in months],
                      showfliers=False, patch_artist=True)
for p in bp2["boxes"]: p.set_facecolor("#e3d8cf")
axes[1].set_xlabel("month"); axes[1].grid(alpha=0.3, axis="y")
axes[1].set_title("by month (all hours pooled)", fontsize=10, loc="left")
fig.suptitle("How much does tau* actually vary?", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(FIGDIR / "clct_tau_regime_summary.png", dpi=120)
plt.show()
print("wrote diagnostics and summary figures")

## 5. Summary

In [ ]:
print("=" * 86)
print(f"Optimal CLCT threshold, KENDA-CH1 det FG vs NWCSAF, "
      f"{TAU.index[0]:%d %b %Y} - {TAU.index[-1]:%d %b %Y}")
print("=" * 86)
print(f"{len(TAU)} timesteps at {SYNOPTIC_HOURS} UTC; {len(failed)} missing\n")
print(f"tau* over everything: median {TAU.tau_star.median():.1f}%, "
      f"IQR {TAU.tau_star.quantile(.25):.1f}-{TAU.tau_star.quantile(.75):.1f}%, "
      f"full range {TAU.tau_star.min():.1f}-{TAU.tau_star.max():.1f}%")
print(f"typical plateau width (J within {PLATEAU_TOL}): "
      f"{(TAU.plateau_hi - TAU.plateau_lo).median():.1f} points -- "
      f"tau* is not pinned to better than this")

cl = TAU[TAU.hour.isin([0, 12])]
print(f"\nThe two CLEAN hours (single regime all year):")
for h in (0, 12):
    s = TAU[TAU.hour == h]
    print(f"  {HOUR_LABEL[h]} ({s.regime.mode()[0]}): median {s.tau_star.median():.1f}%, "
          f"range {s.tau_star.min():.1f}-{s.tau_star.max():.1f}%, "
          f"monthly medians {s.tau_star.resample('MS').median().min():.0f}-"
          f"{s.tau_star.resample('MS').median().max():.0f}%")
print(f"  00Z minus 12Z, on the monthly medians: "
      f"{(TAU[TAU.hour==0].tau_star.resample('MS').median() - TAU[TAU.hour==12].tau_star.resample('MS').median()).mean():+.1f} points")
print(f"\nDWD's 12.5% (1 okta) sits at the {100*(TAU.tau_star < 12.5).mean():.1f}th percentile "
      f"of the fitted values.")
print("\nCaveats: hours are not independent; tau* from a single scene is weakly")
print("determined (see the plateau); and 06/18 UTC change illumination regime")
print("through the year, so part of any seasonal arc there is the satellite.")

## 6. What `tau` actually is

Three things, in one figure:

**(a)** `tau` is nothing more than the crossover between two piles — ICON's CLCT where the satellite
says cloudy, and where it says clear. The piles barely overlap, which is why the comparison works at
all; the width of the overlap is exactly why `tau` cannot be pinned down better than ~19 points.

**(b)** Over two years both piles slide up and down together, and `tau` rides along. That is the whole
seasonal signal.

**(c)** Meanwhile the *distance* between the piles, and the skill `J`, barely move. So the model does
not get better or worse at telling cloud from clear through the year — its cloud field just sits at a
different level. `tau` is a location statistic, not a skill one.

The practical consequence: **`M0` and `M1` say everything `tau` says, and need no threshold at all.**
`M0` in particular -- how much cloud ICON puts where the satellite sees none -- is a direct statement
about the bias this project is chasing, which `tau` never was.

In [ ]:
import matplotlib.dates as mdates

# --- the three quantities, per scene, no threshold involved -----------------
M0, M1, SEP = [], [], []
for i in range(len(TIMES_used)):
    c = CLCT_ALL[i].astype("float64"); o = OBS_ALL[i]
    v = IN_DOMAIN & (o >= 0) & np.isfinite(c)
    if v.sum() < 1000:
        M0.append(np.nan); M1.append(np.nan); SEP.append(np.nan); continue
    cc, oo = c[v], (o[v] == 1)
    a = np.median(cc[~oo]) if (~oo).any() else np.nan
    b = np.median(cc[oo]) if oo.any() else np.nan
    M0.append(a); M1.append(b); SEP.append(b - a)
TAU["M0"], TAU["M1"], TAU["separation"] = M0, M1, SEP

G = TAU[(TAU.obs_cloud_frac >= 0.05) & (TAU.obs_cloud_frac <= 0.95)].copy()

def pick_typical(hour, regime):
    """The most REPRESENTATIVE scene for an hour/regime, not an arbitrary date.
    Hand-picking dates gave a 'winter night' example with M0 = 79% against a night
    median of 23% -- an outlier presented as typical. This takes the scene closest
    to the regime's own median M0, M1 and tau*, each scaled by that quantity's
    spread so no one of them dominates the choice."""
    s = G[(G.hour == hour) & (G.regime == regime)].dropna(subset=["M0", "M1", "tau_star"])
    if not len(s):
        return None
    z = sum(((s[k] - s[k].median()) / max(s[k].std(), 1e-9)) ** 2
            for k in ("M0", "M1", "tau_star"))
    return TIMES_used.index(z.idxmin().to_pydatetime())

fig = plt.figure(figsize=(15, 11))
gs = fig.add_gridspec(3, 2, height_ratios=[1.15, 1.0, 0.85], hspace=0.42, wspace=0.18)

# --- (a) the two piles, for a winter night and a summer midday --------------
for k, (hh, rg, lab) in enumerate([(0, "night", "typical NIGHT scene (00 UTC)"),
                                   (12, "day", "typical DAY scene (12 UTC)")]):
    i = pick_typical(hh, rg)
    if i is None:
        continue
    c = CLCT_ALL[i].astype("float64"); o = OBS_ALL[i]
    v = IN_DOMAIN & (o >= 0) & np.isfinite(c)
    cc, oo = c[v], (o[v] == 1)
    ax = fig.add_subplot(gs[0, k])
    bins = np.linspace(0, 100, 51)
    ax.hist(cc[~oo], bins=bins, density=True, color="#E8A33D", alpha=0.6,
            label="satellite says CLEAR")
    ax.hist(cc[oo], bins=bins, density=True, color="#2C5985", alpha=0.6,
            label="satellite says CLOUDY")
    row = TAU.loc[TIMES_used[i]]
    ax.axvline(row.tau_star, color="k", lw=2.2, zorder=5)
    ax.axvspan(row.plateau_lo, row.plateau_hi, color="k", alpha=0.10, lw=0, zorder=1)
    ax.axvline(row.M0, color="#9A6A1A", ls="--", lw=1.6)
    ax.axvline(row.M1, color="#16324F", ls="--", lw=1.6)
    ax.set_title(f"{lab}   {TIMES_used[i]:%d %b %Y}\n"
                 f"M0 {row.M0:.0f}%   M1 {row.M1:.0f}%   "
                 f"tau* {row.tau_star:.0f}%   J {row.J:.2f}", fontsize=9.5, loc="left")
    ax.set_xlabel("ICON CLCT [%]"); ax.set_xlim(0, 100)
    # ICON's CLCT is strongly bimodal -- nearly every cell is near 0 or near 100.
    # On a linear axis those two spikes flatten everything else to invisibility,
    # hiding the overlap region this panel exists to show. Log y keeps the spikes
    # honest AND makes the overlap readable.
    ax.set_yscale("log")
    ax.set_yticks([])
    ax.set_ylabel("share of cells (log)" if k == 0 else "")
    ax.legend(fontsize=8, loc="upper center", framealpha=0.9)
    ax.annotate("tau*", xy=(row.tau_star, ax.get_ylim()[0] * 3),
                xytext=(6, 0), textcoords="offset points", fontsize=9,
                fontweight="bold", va="bottom")
    if k == 0:
        ax.annotate("grey = thresholds that\nscore just as well",
                    xy=(row.plateau_lo, ax.get_ylim()[0] * 30),
                    xytext=(-5, 0), textcoords="offset points", fontsize=7.5,
                    ha="right", va="bottom")

# --- (b) both piles slide; tau rides along ---------------------------------
ax = fig.add_subplot(gs[1, :])
mm = G.resample("MS")[["M0", "M1", "tau_star"]].median()
ax.fill_between(mm.index, mm.M0, mm.M1, color="#c8d3de", alpha=0.75, lw=0,
                label="between M0 and M1 (where cloudy and clear ICON values live)")
ax.plot(mm.index, mm.M1, "-", color="#16324F", lw=2.0, label="M1  ICON where satellite says CLOUDY")
ax.plot(mm.index, mm.M0, "-", color="#9A6A1A", lw=2.0, label="M0  ICON where satellite says CLEAR")
ax.plot(mm.index, mm.tau_star, "-", color="k", lw=2.4, label="tau*  (rides between them)")
ax.set_ylabel("ICON CLCT [%]"); ax.set_ylim(0, 100); ax.grid(alpha=0.3)
ax.legend(fontsize=8.5, ncol=2, loc="center left")
ax.set_title("(b) both distributions slide up and down together -- tau* simply follows them",
             fontsize=10, loc="left")

# --- (c) but the gap between them, and the skill, barely move --------------
ax = fig.add_subplot(gs[2, :])
ms = G.resample("MS")[["separation", "J"]].median()
ax.plot(ms.index, ms.separation, "-o", color="#3C7A5A", lw=2.0, ms=3.5,
        label="separation M1 - M0 [points]")
ax.set_ylabel("separation [points]"); ax.set_ylim(0, 100); ax.grid(alpha=0.3)
ax2 = ax.twinx()
ax2.plot(ms.index, ms.J, "-s", color="#9B3B3B", lw=2.0, ms=3.5, label="J (skill)")
ax2.set_ylabel("J"); ax2.set_ylim(0, 1)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8.5, loc="lower left", ncol=2)
ax.set_title("(c) ...while the distance between them, and the skill, stay flat all year. "
             "The model is not getting better or worse -- its cloud field is just moving.",
             fontsize=10, loc="left")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))

fig.suptitle("What the CLCT threshold actually measures", fontsize=13)
fig.savefig(FIGDIR / "clct_tau_explained.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"wrote {FIGDIR / 'clct_tau_explained.png'}\n")
print(f"{'':>10s} {'M0':>7s} {'M1':>7s} {'separation':>11s} {'J':>7s}")
for rg in ("day", "twilight", "night"):
    s = G[G.regime == rg]
    print(f"{rg:>10s} {s.M0.median():6.1f}% {s.M1.median():6.1f}% "
          f"{s.separation.median():10.1f} {s.J.median():7.3f}")
print(f"\nseparation over the whole record: median {G.separation.median():.1f} points, "
      f"IQR {G.separation.quantile(.25):.1f}-{G.separation.quantile(.75):.1f}")
print(f"J over the whole record:          median {G.J.median():.3f}, "
      f"IQR {G.J.quantile(.25):.3f}-{G.J.quantile(.75):.3f}")
print("M0 is the informative one: it is how much cloud ICON puts where the satellite")
print("sees none, needs no threshold, and is a direct statement about the bias.")